# Paper 1 (brain paper): figures

Single-column figures (8.7 cm, saved at exactly that width, 400 dpi) from `paper/data/v3/` and `paper/data/p1/` (run `paper1_analysis.ipynb` first). Written to `paper/figures/p1/`. Cohort colours are v1's.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

REPO = Path.cwd().resolve().parent                       # this notebook lives in paper/
DATA, D3, D1, OUT = REPO / "paper/data", REPO / "paper/data/v3", REPO / "paper/data/p1", REPO / "paper/figures/p1"
OUT.mkdir(parents=True, exist_ok=True)
W1 = 3.42                                                # PNAS single column, 8.7 cm; figures are saved at exactly this width
INK, INK2 = "#1a1a1a", "#666666"
ORDER = ["dHCP", "BCP", "CALM", "RED", "ACE", "HCPd", "HCPya", "camCAN", "HCPa"]
COH = dict(zip(ORDER, ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#7f7f7f", "#bcbd22", "#17becf"]))  # cohort colours of v1

sns.set_theme(context="paper", style="ticks")
mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "mathtext.fontset": "stix", "font.size": 7.5, "axes.labelsize": 7.5, "axes.titlesize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 6.8, "legend.title_fontsize": 6.8,
    "axes.linewidth": 0.6, "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "text.color": INK, "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.minor.width": 0.45,
    "ytick.minor.width": 0.45, "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.minor.size": 1.4,
    "ytick.minor.size": 1.4, "xtick.major.pad": 1.6, "ytick.major.pad": 1.6, "axes.labelpad": 1.8,
    "lines.linewidth": 1.0, "legend.frameon": False, "legend.handlelength": 1.3, "legend.handletextpad": 0.4,
    "legend.borderaxespad": 0.2, "legend.labelspacing": 0.25, "pdf.fonttype": 42, "savefig.dpi": 400})


def letter(ax, s, x=-0.3, y=1.0):
    ax.text(x, y, s, transform=ax.transAxes, fontsize=9.5, fontweight="bold", va="bottom", ha="left")


def save(fig, name):
    """Exact figure size (no tight bounding box), so the text keeps its size in the column."""
    for ext in ("pdf", "png"):
        fig.savefig(OUT / f"{name}.{ext}", dpi=400)


def r_(a, b):
    return float(np.corrcoef(a, b)[0, 1])


def age_means(d, col):
    """Mean of col per cohort and age: 1-year bins, 3-month bins below 2 years (as in v1)."""
    a = np.where(d.age < 2, np.floor(d.age * 4) / 4 + 0.125, np.floor(d.age) + 0.5)
    return d.assign(abin=a).groupby(["cohort", "abin"])[col].mean().reset_index()


def trajectories(ax, d, col, ylabel):
    m = age_means(d, col)
    for c in ORDER:
        g = m[m.cohort == c]
        ax.plot(g.abin, g[col], "-o", color=COH[c], ms=1.7, lw=0.55, mew=0)
    ax.set(xlabel="age (years)", ylabel=ylabel, xlim=(-2, 92), xticks=[0, 20, 40, 60, 80])

In [ ]:
d = pd.read_csv(D3 / "subjects_v3.csv"); N = json.load(open(D3 / "numbers_v3.json"))      # structure, simulated MC
N1 = json.load(open(D1 / "numbers_p1.json")); race = pd.read_csv(D1 / "race_metrics.csv")
nodes = pd.read_csv(D1 / "cii_nodes_sample.csv"); orders = pd.read_csv(D1 / "cii_orders_by_degree.csv")
nl = pd.read_csv(D1 / "nulls_sim.csv"); kap = pd.read_csv(D1 / "kappa_sim.csv")
uk = pd.read_csv(D3 / "upsilon_k.csv"); nodes_k = pd.read_csv(D3 / "nodes_sample.csv")
FX = mpl.ticker.FixedLocator; SF = mpl.ticker.ScalarFormatter

## Subgraph centrality and the orders of its walk expansion (replaces Table 1)
(a) $C_{ii}$ against its order-2 truncation, nodes of 150 random connectomes; (b) fraction of $C_{ii}-1$ reproduced up to order 2, 3 and 4, by degree.

In [ ]:
fig = plt.figure(figsize=(W1, 1.9))
gs = fig.add_gridspec(1, 2, left=0.135, right=0.985, top=0.93, bottom=0.215, wspace=0.52)
# (a) subgraph centrality of each node against its order-2 truncation, coloured by degree
ax = fig.add_subplot(gs[0])
sc = ax.scatter(nodes.ord2, nodes.C, c=nodes.k, cmap="viridis", vmin=0, vmax=60, s=0.6, lw=0, alpha=0.6, rasterized=True)
lo, hi = nodes.C.min(), nodes.C.max(); ax.plot([lo, hi], [lo, hi], color=INK2, lw=0.6, ls="--")
ax.text(0.05, 0.95, f"$R^2$ = {N1['cii_nodes']['R2_order2']:.4f}", transform=ax.transAxes, va="top")
cax = ax.inset_axes([0.5, 0.12, 0.45, 0.05]); cb = fig.colorbar(sc, cax=cax, orientation="horizontal")
cb.set_ticks([0, 30, 60]); cb.ax.tick_params(labelsize=5.5, length=1.5, pad=1, width=0.4); cb.outline.set_linewidth(0.3)
cax.set_title("degree $k$", fontsize=6, pad=1.5)
ax.set(xlabel="order 2: $1+\\tilde s_i^2Y_i/2$", ylabel="$C_{ii}=(e^{\\tilde W})_{ii}$"); letter(ax, "a")
# (b) fraction of C_ii - 1 reproduced by the expansion truncated at order n, as a function of degree
ax = fig.add_subplot(gs[1]); kc = np.sqrt(orders.k_lo * orders.k_hi); ok = orders.n_nodes >= 100
for n, col in ((2, "#9ecae1"), (3, "#4292c6"), (4, "#08306b")):
    ax.plot(kc[ok], orders[f"frac{n}"][ok], "-o", color=col, ms=2.2, mew=0, lw=1, label=f"orders $\\leq${n}")
ax.set(xscale="log", xlabel="degree $k$", ylabel="fraction of $C_{ii}-1$", ylim=(0.885, 1.005))
ax.xaxis.set_major_locator(FX([3, 10, 30])); ax.xaxis.set_major_formatter(SF()); ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
ax.legend(loc="lower left"); letter(ax, "b")
for a in fig.axes:
    if a is not cax:
        sns.despine(ax=a)
save(fig, "fig_cii_orders")

## What predicts memory
(a) MC against the stable rank; (b) single walk orders and $\langle C_{ii}\rangle$ truncated at each order; (c) predictors ordered from the topology to the whole linear response and the spectrum (bars: pooled $r$; line: range within cohorts, dot: median; diamonds: partial $r$ given the stable rank); (d) factor shares of the stable rank; (e) covariance of the linear response vs coupling, and the controllability Gramian; (f) null models.

In [ ]:
fig = plt.figure(figsize=(W1, 6.8))
gs = fig.add_gridspec(4, 2, left=0.135, right=0.985, top=0.975, bottom=0.06, hspace=0.72, wspace=0.55, height_ratios=[1, 1.5, 1, 1.05])
R = N1["race"]
# (a) memory against the stable rank, coloured by cohort
ax = fig.add_subplot(gs[0, 0])
for c in ORDER:
    g = d[d.cohort == c]; ax.scatter(g.rs, g.MC, s=0.8, color=COH[c], lw=0, alpha=0.55, rasterized=True)
b = np.polyfit(np.log(d.rs), d.MC, 1); xs = np.geomspace(d.rs.min(), d.rs.max(), 50); ax.plot(xs, np.polyval(b, np.log(xs)), color=INK, lw=0.9)
ax.set_xscale("log"); ax.set_xticks([3, 5, 10]); ax.xaxis.set_major_formatter(SF()); ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax.text(0.05, 0.95, f"$r$ = {R['rs']['r']:.2f}", transform=ax.transAxes, va="top")
ax.set(xlabel="stable rank $r_s$", ylabel="memory capacity", yticks=[8, 10, 12]); letter(ax, "a")
# (b) which walk order carries memory: single orders, and subgraph centrality truncated at each order
ax = fig.add_subplot(gs[0, 1]); n = np.arange(2, 9)
ax.bar(n, [R[f"tr{m}"]["r"] for m in n], width=0.65, color=["#08519c"] + ["#9ecae1"] * 6, edgecolor=INK, lw=0.3)
ax.plot(n, [R[f"cii_ord{m}"]["r"] for m in n], "-", color="#e6550d", lw=1.1)
ax.text(8.45, R["cii_ord8"]["r"] + 0.05, "$\\langle C_{ii}\\rangle$ up to order $n$", color="#e6550d", fontsize=6, ha="right", va="bottom")
ax.axhline(0, color=INK2, lw=0.5)
ax.set(xlabel="walk order $n$ of $\\mathrm{Tr}\\,\\tilde W^n$", ylabel="$r$ with MC", xticks=list(n), ylim=(-0.15, 1.12)); letter(ax, "b")
# (c) predictors with a reason: from the topology to the whole linear response, and the spectrum
pos = gs[1, :].get_position(fig); ax = fig.add_axes([0.37, pos.y0, 0.985 - 0.37, pos.height])
items = [("mean degree", "k", "topology"), ("strength factor $G$", "G", "order 2"), ("disparity ratio", "Ratio", "order 2"),
         ("disparity factor $L$", "L", "order 2"), ("stable rank $r_s$", "rs", "order 2"), ("order 3 ($\\mathrm{Tr}\\,\\tilde W^3$)", "tr3", "higher orders"), ("order 4 ($\\mathrm{Tr}\\,\\tilde W^4$)", "tr4", "higher orders"),
         ("subgraph centrality", "cii", "all orders"), ("communicability", "comm", "all orders"), ("linear attenuation", "lam", "linear response"),
         ("SAR covariance", "sar0.5", "linear response"), ("average controllability", "avgctrl", "linear response"), ("eigenvalue entropy", "spec_entropy", "spectrum")]
GC = {"topology": "#969696", "order 2": "#08519c", "higher orders": "#6baed6", "all orders": "#e6550d", "linear response": "#756bb1", "spectrum": "#31a354"}
y = np.arange(len(items))[::-1]
for yi, (lab, key, grp) in zip(y, items):
    r = R[key]; ax.barh(yi, r["r"], height=0.62, color=GC[grp], alpha=0.85 if key != "rs" else 1, edgecolor=INK, lw=0.3)
    ax.plot([r["within_min"], r["within_max"]], [yi, yi], color=INK, lw=0.7); ax.plot(r["within_median"], yi, "o", ms=2.2, color=INK)
    ax.plot(r["partial"], yi, "D", ms=3, mfc="white", mec=INK, mew=0.6)
ax.set_yticks(y); ax.set_yticklabels([i[0] for i in items], fontsize=6.4); ax.tick_params(axis="y", length=0)
ax.axvline(0, color=INK2, lw=0.5); ax.set(xlabel="correlation with memory capacity", xlim=(-0.75, 1.02))
hs = [mpl.patches.Patch(facecolor=GC[g], edgecolor=INK, lw=0.3, label=g) for g in GC] + \
     [mpl.lines.Line2D([], [], color=INK, lw=0.7, marker="o", ms=2.2, label="within cohorts"),
      mpl.lines.Line2D([], [], ls="", marker="D", ms=3, mfc="white", mec=INK, label="given $r_s$")]
ax.legend(handles=hs, loc="lower center", bbox_to_anchor=(0.31, 1.0), ncol=4, fontsize=5.6, columnspacing=0.6, handlelength=1.0, borderaxespad=0.25); letter(ax, "c", x=-0.58)
# (d) which factor of the stable rank varies between people (exact shares of var log r_s)
ax = fig.add_subplot(gs[2, 0]); fs = N["factor_shares"]; rows = ["all"] + ORDER
for f, mk, col in (("G", "o", INK), ("D", "s", "#a3a3a3"), ("L", "^", "#e6550d")):
    ax.scatter([fs[r][f] for r in rows], np.arange(len(rows)), marker=mk, s=10, color=col, lw=0, label=f"${f}$", zorder=3)
ax.axvline(0, color=INK2, lw=0.5); ax.axvline(1, color=INK2, lw=0.5, ls=":")
ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, fontsize=6); ax.invert_yaxis(); ax.tick_params(axis="y", length=0, pad=1)
for t, r in zip(ax.get_yticklabels(), rows):
    t.set_color(COH.get(r, INK)); t.set_fontweight("bold" if r == "all" else "normal")
ax.set(xlabel="share of var(log $r_s$)", xlim=(-0.65, 1.8), xticks=[0, 1])
ax.legend(loc="lower center", bbox_to_anchor=(0.45, 0.97), ncol=3, columnspacing=1.0, handletextpad=0.05); letter(ax, "d", x=-0.36)
# (e) covariance of the linear response: predictive power as the response approaches criticality
ax = fig.add_subplot(gs[2, 1]); al = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
ax.plot(al, [R[f"sar{a}"]["r"] for a in al], "-o", color="#756bb1", ms=2.5, mew=0, lw=1, label="SAR covariance")
ax.plot([1.0], [R["avgctrl"]["r"]], "s", color=INK, ms=3.5, label="Gramian (avg. control.)")
ax.axhline(R["rs"]["r"], color="#08519c", lw=0.8, ls="--"); ax.text(0.27, R["rs"]["r"] + 0.03, "stable rank", color="#08519c", fontsize=6)
ax.axhline(0, color=INK2, lw=0.5)
ax.set(xlabel="coupling $\\alpha$ (1 = critical)", ylabel="$r$ with MC", ylim=(-0.3, 1.1), xticks=[0.25, 0.5, 0.75, 1.0])
ax.legend(loc="center left", fontsize=5.8); letter(ax, "e")
# (f) null models (simulated, 10 realisations): observed change in memory and the change predicted by the stable rank
ax = fig.add_subplot(gs[3, :])
names = {"Uniform": "Uniform", "BrokenStick": "Broken\nstick", "Reshuffle": "Reshuffle", "EqualStrength": "Equal\nstrength", "InterHalf": "Interhem.\nweights ×½"}
NC = dict(zip(names.values(), ["#e6550d", "#fdae6b", "#969696", "#3182bd", "#756bb1"]))
q = nl.assign(null=nl.model.map(names)); lab = list(names.values()); ax.axhline(0, color=INK2, lw=0.5)
sns.boxplot(data=q, x="null", y="dMC", order=lab, color="white", width=0.42, fliersize=0, linewidth=0.6, ax=ax,
            boxprops=dict(edgecolor=INK2), whiskerprops=dict(color=INK2), medianprops=dict(color=INK), capprops=dict(color=INK2))
sns.stripplot(data=q, x="null", y="dMC", order=lab, hue="null", palette=NC, size=2, jitter=0.17, alpha=0.8, ax=ax, legend=False, rasterized=True)
pred = q.groupby("null").dMC_pred.median().reindex(lab)
ax.scatter(np.arange(len(lab)), pred.values, marker="D", s=18, color=INK, edgecolor="white", lw=0.5, zorder=5, label="predicted from $r_s$")
ax.set(xlabel="", ylabel="$\\Delta$MC (null $-$ real)"); ax.tick_params(axis="x", length=0); ax.legend(loc="upper left"); letter(ax, "f", x=-0.135)
for a in fig.axes:
    sns.despine(ax=a)
save(fig, "fig_memory_predictors")

## Lifespan and aging (as v3 Fig. 3; virtual aging now simulated)

In [ ]:
fig = plt.figure(figsize=(W1, 6.1))
outer = fig.add_gridspec(1, 2, left=0.135, right=0.985, top=0.975, bottom=0.112, wspace=0.52)
gl = outer[0].subgridspec(2, 1, height_ratios=[1.25, 3.75], hspace=0.16)
gla = gl[1].subgridspec(4, 1, hspace=0.3)
gr = outer[1].subgridspec(3, 1, height_ratios=[1.3, 1.15, 1.3], hspace=0.34)

# (a) nodal disparity against degree (v1 Fig. 4a)
ax = fig.add_subplot(gl[0]); ax_a = ax
ax.scatter(nodes_k.k, nodes_k.Ups, s=0.5, color="#6baed6", alpha=0.18, lw=0, rasterized=True)
k = np.arange(1, 90.0); mu = 2 * k / (k + 1)
sd = k * np.sqrt((20 + 4 * k) / ((k + 1) * (k + 2) * (k + 3)) - 4 / (k + 1) ** 2)
ax.fill_between(k, np.clip(mu - 2 * sd, 0.7, None), mu + 2 * sd, color="#bdbdbd", alpha=0.8, lw=0, label="null $\\pm2\\sigma$")
ax.plot(k, mu, color="#636363", lw=0.7)
ax.plot(k, k, color="#d62728", lw=0.8, ls="--", label="$\\Upsilon=k$")
g = uk[uk["size"] >= 30]
ax.plot(g.k, g["mean"], color="#08519c", lw=1.3, label=f"$\\langle\\Upsilon\\rangle_k\\propto k^{{{N['upsilon_alpha']:.2f}}}$")
ax.set(xscale="log", yscale="log", xlabel="degree $k$", ylabel="$\\Upsilon=kY$", xlim=(1, 90), ylim=(0.7, 90))
for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_locator(mpl.ticker.FixedLocator([1, 3, 10, 30, 90])); axis.set_major_formatter(mpl.ticker.ScalarFormatter())
    axis.set_minor_locator(mpl.ticker.NullLocator())
ax.legend(loc="upper left", fontsize=6.3); letter(ax, "a")

# (b-e) lifespan trajectories on a shared age axis: per-age means of each cohort
spec = (("Ratio", "$\\langle\\Upsilon/\\Upsilon_{null}\\rangle$"), ("MC", "MC"), ("rs", "$r_s$"), ("hom_F2", "$h$"))
axs = []
for i, (col, lab) in enumerate(spec):
    ax = fig.add_subplot(gla[i], sharex=axs[0] if axs else None); trajectories(ax, d, col, lab)
    if i < 3:
        ax.set_xlabel(""); ax.tick_params(labelbottom=False)
    letter(ax, "bcde"[i]); axs.append(ax)
fig.align_ylabels([ax_a] + axs)

# (f) memory against the homotopic share, coloured by age
ax = fig.add_subplot(gr[0])
sc = ax.scatter(d.hom_F2, d.MC, c=d.age, cmap="viridis", vmin=0, vmax=90, s=0.8, lw=0, alpha=0.65, rasterized=True)
cax = ax.inset_axes([0.5, 0.9, 0.46, 0.05]); cb = fig.colorbar(sc, cax=cax, orientation="horizontal")
cb.set_ticks([0, 45, 90]); cb.ax.tick_params(labelsize=6, length=1.5, pad=1, width=0.4); cb.outline.set_linewidth(0.3)
cax.set_title("age (years)", fontsize=6.3, pad=1.5)
ax.text(0.05, 0.05, f"$r$ = {r_(d.hom_F2, d.MC):.2f}", transform=ax.transAxes)
ax.set(xlabel="homotopic share $h$", ylabel="MC", ylim=(d.MC.min() - 0.2, d.MC.max() + 1.1), yticks=[8, 10, 12])
letter(ax, "f", x=-0.36)

# (g) within camCAN and HCPa: the homotopic share accounts for the aging rise of memory
ax = fig.add_subplot(gr[1]); L = N["lifespan"]; w = 0.27
bars = (("r_age_hom", "$h$", "#6a3d9a"), ("r_age_MC", "MC", INK), ("pr_age_MC_given_hom", "MC | $h$", "#bdbdbd"))
for j, (key, lab, col) in enumerate(bars):
    ax.bar(np.arange(2) + (j - 1) * w, [L[c][key] for c in ("camCAN", "HCPa")], width=w * 0.92, color=col, edgecolor=INK,
           lw=0.3, label=lab)
ax.axhline(0, color=INK, lw=0.5); ax.set_xticks([0, 1]); ax.set_xticklabels(["camCAN", "HCPa"]); ax.tick_params(axis="x", length=0)
for t, c in zip(ax.get_xticklabels(), ("camCAN", "HCPa")):
    t.set_color(COH[c]); t.set_fontweight("bold")
ax.set(ylabel="$r$ with age", ylim=(-0.72, 0.62), xlim=(-0.5, 1.5), yticks=[-0.5, 0, 0.5])
ax.legend(loc="upper center", ncol=3, handlelength=0.8, columnspacing=0.6, handletextpad=0.25); letter(ax, "g", x=-0.36)

# (h) virtual aging: young connectomes with rescaled interhemispheric weights
ax = fig.add_subplot(gr[2]); VR = N1["virtual_rescaling"]
for c in ("camCAN", "HCPa"):
    g = kap[kap.cohort == c].groupby("kappa").MC.agg(["mean", "sem"])
    ax.fill_between(g.index, g["mean"] - 1.96 * g["sem"], g["mean"] + 1.96 * g["sem"], color=COH[c], alpha=0.25, lw=0)
    ax.plot(g.index, g["mean"], "-o", color=COH[c], ms=2, mew=0, lw=1)
    ax.axhline(VR[c]["MC_old"], color=COH[c], lw=0.8, ls="--")
    Fy, Fo = VR[c]["inter_young"], VR[c]["inter_old"]                # interhemispheric fraction of the total weight
    ax.axvline(Fo * (1 - Fy) / (Fy * (1 - Fo)), color=COH[c], lw=0.7, ls=":")   # kappa that brings Fy to Fo
ax.text(1.52, VR["camCAN"]["MC_old"] + 0.03, "≥ 70 y, observed", ha="right", va="bottom", fontsize=6.3, color=INK2)
ax.text(0.47, 9.8, "$\\kappa$ matching\n≥ 70 y", ha="left", va="bottom", fontsize=6.3, color=INK2)
ax.set(xlabel="interhem. weights × $\\kappa$", ylabel="MC", xticks=[0.5, 1, 1.5])
letter(ax, "h", x=-0.36)

hs = [mpl.lines.Line2D([], [], color=COH[c], marker="o", ms=3, lw=0.7, label=c) for c in ORDER]
fig.legend(handles=hs, loc="lower center", bbox_to_anchor=(0.54, 0.0), ncol=5, columnspacing=0.8, handletextpad=0.3, fontsize=6.8)
for a in fig.axes:
    if a is not cax:
        sns.despine(ax=a)
save(fig, "fig_lifespan_aging")

## SI: development, stated honestly
(a) Age trends within each cohort; (b) turning point of memory within camCAN; (c, d) young adults of HCPya and camCAN at the same ages.

In [ ]:
fig = plt.figure(figsize=(W1, 4.3))
gs = fig.add_gridspec(2, 2, left=0.16, right=0.985, top=0.97, bottom=0.09, hspace=0.5, wspace=0.55, height_ratios=[1.15, 1])
W = N1["within"]; coh = [c for c in ORDER if f"{c}|MC" in W]
# (a) age trends within each cohort (95% CI)
ax = fig.add_subplot(gs[0, 0]); y = np.arange(len(coh))[::-1]
for off, col, key, lab in ((0.16, INK, "MC", "MC"), (-0.16, "#6a3d9a", "hom_F2", "$h$")):
    for yi, c in zip(y, coh):
        w = W[f"{c}|{key}"]; ax.plot([w["lo"], w["hi"]], [yi + off] * 2, color=col, lw=0.8); ax.plot(w["r"], yi + off, "o", color=col, ms=2.6)
    ax.plot([], [], "o-", color=col, ms=2.6, lw=0.8, label=lab)
ax.axvline(0, color=INK2, lw=0.5); ax.set_yticks(y); ax.set_yticklabels(coh, fontsize=6.2); ax.tick_params(axis="y", length=0)
for t, c in zip(ax.get_yticklabels(), coh): t.set_color(COH[c])
ax.set(xlabel="correlation with age\nwithin the cohort", xlim=(-0.75, 0.75)); ax.legend(loc="upper right"); letter(ax, "a", x=-0.45)
# (b) the turning point of memory within camCAN (single site)
ax = fig.add_subplot(gs[0, 1]); cc = d[d.cohort == "camCAN"]; tp = N1["camcan_turning_point"]["MC"]
ax.scatter(cc.age, cc.MC, s=1.2, color=COH["camCAN"], alpha=0.45, lw=0, rasterized=True)
bm = cc.groupby(pd.cut(cc.age, np.arange(15, 95, 5)), observed=True).agg(a=("age", "mean"), m=("MC", "mean"), s=("MC", "sem"))
ax.errorbar(bm.a, bm.m, yerr=1.96 * bm.s, color=INK, lw=0.9, marker="o", ms=2, capsize=1)
ax.axvspan(*tp["ci"], color="#d9d9d9", alpha=0.6, lw=0); ax.axvline(tp["age"], color=INK, lw=0.7, ls="--")
ax.text(tp["age"] + 1, ax.get_ylim()[1], f"{tp['age']:.0f} y", va="top", fontsize=6.3)
ax.set(xlabel="age (years)", ylabel="MC (camCAN)", xticks=[20, 40, 60, 80]); letter(ax, "b")
# (c) young adults of two cohorts at the same ages
for j, (col, lab) in enumerate((("MC", "memory capacity"), ("hom_F2", "homotopic share $h$"))):
    ax = fig.add_subplot(gs[1, j]); m = d[(d.age >= 22) & (d.age <= 37) & d.cohort.isin(["HCPya", "camCAN"])]
    sns.boxplot(data=m, x="cohort", y=col, order=["HCPya", "camCAN"], color="white", width=0.45, fliersize=0, linewidth=0.6, ax=ax,
                boxprops=dict(edgecolor=INK2), whiskerprops=dict(color=INK2), medianprops=dict(color=INK), capprops=dict(color=INK2))
    sns.stripplot(data=m, x="cohort", y=col, order=["HCPya", "camCAN"], hue="cohort", palette=COH, size=1.4, jitter=0.18, alpha=0.6, ax=ax, legend=False, rasterized=True)
    ax.set(xlabel="ages 22-37", ylabel=lab); ax.tick_params(axis="x", length=0); letter(ax, "c" if j == 0 else "d", x=-0.38)
for a in fig.axes:
    sns.despine(ax=a)
save(fig, "figS_development")